In [18]:
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM

In [19]:
# ==========================================
# 1. MODEL
# ==========================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [20]:
# ==========================================
# 2. TOKENIZER
# ==========================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

Loading tokenizer...


In [21]:
# ==========================================
# 3. TRANSFORMER MODEL
# ==========================================

print("Loading Transformer model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading Transformer model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!


In [22]:
# ==========================================
# 4. CHATBOT FUNCTION
# ==========================================

def chatbot(message, history):

    # ------------------------------
    # System instructions
    # ------------------------------

    messages = [
        {
            "role": "system",
            "content": """
You are a helpful Coding Assistant AI.

Help users write, explain, debug, and fix code
in Python and other programming languages.

Always give clear, well-formatted code with
proper indentation, and explain your solution
in simple, easy-to-understand language.

If the user shares code with an error, point out
the exact issue and give the corrected version.

Focus on programming, algorithms, data structures,
and software development topics.
"""
        }
    ]


    # ------------------------------
    # Add previous conversation
    # ------------------------------

    for chat in history:

        if chat["role"] in ["user", "assistant"]:

            messages.append({
                "role": chat["role"],
                "content": chat["content"]
            })


    # ------------------------------
    # Add current user message
    # ------------------------------

    messages.append({
        "role": "user",
        "content": message
    })


    # ==========================================
    # PHASE 1
    # CHAT TEMPLATE / TOKENIZATION PREPARATION
    # ==========================================

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # ==========================================
    # PHASE 2
    # TEXT → TOKEN IDs
    # ==========================================

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    # ==========================================
    # PHASE 3 + PHASE 4
    # TRANSFORMER + NEXT TOKEN PREDICTION
    # ==========================================

    with torch.no_grad():

        output_ids = model.generate(

            **inputs,

            max_new_tokens=400,

            temperature=0.4,

            top_p=0.85,

            do_sample=True,

            repetition_penalty=1.1
        )


    # ==========================================
    # GET ONLY NEW TOKENS
    # ==========================================

    new_tokens = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]


    # ==========================================
    # PHASE 5
    # TOKEN IDs → HUMAN TEXT
    # ==========================================

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return response

In [23]:
# ==========================================
# 5. GRADIO USER INTERFACE
# ==========================================

demo = gr.ChatInterface(

    fn=chatbot,

    title="💻 My Coding Helper Bot",

    description="""
    An AI coding assistant built using
    Hugging Face Transformers and Gradio.
    Helps you write, explain, and debug code.
    """,

    examples=[
        "Write Python code for a calculator.",
        "Explain what a for loop does.",
        "Fix this code: for i in range(10) print(i)",
        "What is the difference between a list and a tuple?",
        "Write a function to check if a number is prime."
    ],

    save_history=True
)

In [24]:
# ==========================================
# 6. START APPLICATION
# ==========================================

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d981d89800345f25b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
